# DSCI 6015 Midterm Project: Cloud-Based PE Malware Detection

## Task 1: Building and Training MalConv in PyTorch

This notebook implements a deep learning malware classifier based on the **MalConv architecture**, trained on the **EMBER 2017 v2 dataset**.

Rather than using pre-vectorized files, this implementation performs **feature vectorization from scratch** using the raw EMBER JSONL files, giving us full visibility into the preprocessing pipeline

### Overview of Steps
1. Install dependencies
2. Verify GPU and configure local paths
3. Vectorize raw EMBER JSONL files --> `.dat` files
4. Load, filter, and preprocess the feature vectors
5. (Optional) Subsample the dataset
6. Standardize features
7. Define the MalConv model architecture in PyTorch
8. Prepare DataLoaders
9. Train the model with validation tracking and checkpointing
10. Evaluate on the test set
11. Save & load model weights
12. Define an end-to-end inference function


## Step 1: Install Dependencies

Libraries to install:
- ember - PFGimenez fork of the EMBER library, which fixes compatibility issues with newer versions of lief
- lief - binary analysis library that ember uses internally to parse PE file structures
- seaborn - to plot confusion matrix

In [ ]:
import sys
!{sys.executable} -m pip install git+https://github.com/PFGimenez/ember.git -q
!{sys.executable} -m pip install lief seaborn -q
print("Dependencies installed.")

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

## Step 2: Verify GPU and Configure Paths

Verify PyTorch can see the NVIDIA GPU with CUDA. Alternatively use CPU.

- DATA_DIR - the folder containing your extracted EMBER 2017 v2 .jsonl files
- SAVE_DIR - where model checkpoints and outputs will be written

In [ ]:
import torch
import os

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected.")

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = './ember_2017_2'  # JSONL source files
SAVE_DIR = './malconv_output'  # checkpoints & .dat files
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"\nData directory: {DATA_DIR}")
print(f"Save directory: {SAVE_DIR}")
print(f"JSONL files found: {[f for f in os.listdir(DATA_DIR) if f.endswith('.jsonl')]}")


## Step 3: Vectorize Raw EMBER Features

The EMBER 2017 v2 dataset ships as JSONL files (train_features_0.jsonl through train_features_5.jsonl, plus test_features.jsonl). Each line is a JSON object representing hand-crafted features extracted from one PE file including byte histograms, header fields, section info, import tables, and more.

ember.create_vectorized_features() reads all JSONL files and converts each JSON feature dict into a fixed-length float32 vector of 2,381 dimensions, writing binary .dat files to the same directory. We then move those files into SAVE_DIR to keep the source data folder clean.

If .dat files already exist, it will skip creating them

In [ ]:
import ember

dat_files = ['X_train.dat', 'y_train.dat', 'X_test.dat', 'y_test.dat']
already_vectorized = all(os.path.exists(os.path.join(SAVE_DIR, f)) for f in dat_files)

if already_vectorized:
    print("Vectorized .dat files exist, skipping vectorization.")
else:
    print("Starting vectorization...")
    # Vectorize reads from DATA_DIR (JSONL) and writes .dat files there,
    # then we move them into SAVE_DIR to keep things organized
    ember.create_vectorized_features(DATA_DIR)
    ember.create_metadata(DATA_DIR)
    import shutil
    for fname in ['X_train.dat', 'y_train.dat', 'X_test.dat', 'y_test.dat', 'metadata.csv']:
        src_path = os.path.join(DATA_DIR, fname)
        dst_path = os.path.join(SAVE_DIR, fname)
        if os.path.exists(src_path):
            shutil.move(src_path, dst_path)
            print(f"Moved {fname} -> {SAVE_DIR}")
    print("Vectorization complete.")


## Step 4: (Optional) Subsample the Dataset

To speed up experimentation, we can draw a stratified random sample that preserves the malware / benign class ratio.

The dataset is imbalanced. There are more benign samples than malware, so naive random sampling would likely skew the distribution. Stratified sampling keeps both classes are proportionally represented in the subsample.

In [ ]:
import numpy as np

# Load vectorized features from SAVE_DIR - memory-mapped, not all loaded into RAM at once
X_train, y_train, X_test, y_test = ember.read_vectorized_features(SAVE_DIR)

# Filter out unlabeled samples (label == -1)
train_mask = y_train != -1
X_train = X_train[train_mask]
y_train = y_train[train_mask]

test_mask = y_test != -1
X_test = X_test[test_mask]
y_test = y_test[test_mask]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nClass distribution (train): {int((y_train==0).sum())} benign, {int((y_train==1).sum())} malware")


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Reduce training set size to make the experiment more conservative
SUBSAMPLE_SIZE = 25_000

X_train, _, y_train, _ = train_test_split(
    X_train,
    y_train,
    train_size=SUBSAMPLE_SIZE,
    random_state=42,
    stratify=y_train
)

print("New training size:", len(X_train))
print("New train benign:", (y_train == 0).sum())
print("New train malware:", (y_train == 1).sum())

In [ ]:
# Data integrity checks: run before scaling or training
import numpy as np
import hashlib

print(f"Train samples: {len(X_train)}")
print(f"Test samples:  {len(X_test)}")

print(f"Train benign:  {(y_train == 0).sum()}")
print(f"Train malware: {(y_train == 1).sum()}")
print(f"Test benign:   {(y_test == 0).sum()}")
print(f"Test malware:  {(y_test == 1).sum()}")

def row_hash(row):
    # Hash one feature vector so we can spot exact duplicate rows
    arr = np.ascontiguousarray(np.asarray(row, dtype=np.float32))
    return hashlib.sha1(arr.tobytes()).hexdigest()

rng = np.random.default_rng(42)
n_check = min(1000, len(X_train), len(X_test))

train_idx = rng.choice(len(X_train), size=n_check, replace=False)
test_idx = rng.choice(len(X_test), size=n_check, replace=False)

train_hashes = {row_hash(X_train[i]) for i in train_idx}
test_hashes = {row_hash(X_test[i]) for i in test_idx}
overlap = train_hashes & test_hashes

print(f"Checked {n_check} random train rows and {n_check} random test rows")
print(f"Exact duplicate hashes found: {len(overlap)}")
if overlap:
    print("Example overlap hash:", next(iter(overlap)))

## Step 5: Preprocessing - Standardization

The 2,381 EMBER features have very different scales (some are byte counts in the millions while others are normalized ratios between 0 and 1). Training a neural network on unscaled features causes some dimensions to dominate the gradient updates, leading to slow or unstable training.

We apply StandardScaler (zero mean, unit variance) to each feature dimension. The full training set is too large to fit in memory at once for fitting. For this, we use partial_fit() in chunks of 100,000 samples. This computes the running mean and variance without loading everything at once.

We fit the scaler only on training data, then apply the same transform to the train and test sets because fitting on test data would constitute data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
chunk_size = 100_000

# Partial fit in chunks to avoid loading all data into RAM
for start in range(0, len(X_train), chunk_size):
    scaler.partial_fit(X_train[start:start + chunk_size])

# Transform both sets using the fitted scaler
# Convert to numpy array first since X_train may be a memmap
X_train = scaler.transform(np.array(X_train, dtype=np.float32))
X_test = scaler.transform(np.array(X_test, dtype=np.float32))

print(f"Scaling complete. X_train dtype: {X_train.dtype}, shape: {X_train.shape}")

## Step 6: Define the MalConv Architecture

The original MalConv (Anderson & Roth, 2018) operates on raw byte sequences using a gated convolutional network. This implementation adapts this for pre-vectorized EMBER features, a 2,381 dimensional float vector per sample, using a 1D CNN.

### Architecture:
- Input: (batch, 1, 2381) - the feature vector treated as a 1D signal with 1 channel
- Conv1 + ReLU: 128 filters, kernel=128, stride=128 - learns local feature patterns
- Conv2 + ReLU: 128 filters, kernel=4, stride=1 - refines representations
- Adaptive Max Pooling: collapses the temporal dimension to a single value per filter
- FC1 + ReLU: 128 --> 64 hidden units
- FC2: 64 --> 1 output logit
- Sigmoid: squashes output to [0, 1] for binary classification probability

The output is a single probability: values > 0.5 are classified as malware, ≤ 0.5 are classified as benign.

In [ ]:
import torch
import torch.nn as nn

class MalConv(nn.Module):
    def __init__(self, input_dim=2381, output_dim=1):
        super(MalConv, self).__init__()

        # Two 1D convolutional layers to extract local patterns from the feature vector
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=128, stride=128)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=128, kernel_size=4, stride=1)

        # Adaptive max pooling collapses variable-length conv output to a fixed size
        self.pool = nn.AdaptiveMaxPool1d(1)

        # Fully connected layers for final classification
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, output_dim)

        
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, 2381)
        # Unsqueeze to add channel dim: (batch_size, 1, 2381)
        x = x.unsqueeze(1)

        # Convolutional feature extraction
        x = self.relu(self.conv1(x))   # -> (batch, 128, 18)
        x = self.relu(self.conv2(x))   # -> (batch, 128, 15)

        # Global max pooling -> (batch, 128, 1) -> (batch, 128)
        x = self.pool(x).squeeze(-1)

        # Fully connected classification head
        x = self.relu(self.fc1(x))     # -> (batch, 64)
        x = self.dropout(x)
        x = self.fc2(x)                # -> (batch, 1)

        # Sigmoid for binary probability output
        x = self.sigmoid(x)
        return x

# Instantiate and inspect the model
model = MalConv()
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

## Step 7: Prepare DataLoaders

PyTorch's DataLoader handles batching, shuffling, and efficient data feeding to the GPU during training. We:
1. Convert our numpy arrays to PyTorch tensors
2. Split training data 80/20 into train and validation sets
3. Wrap them in TensorDataset and DataLoader

The validation set is held out during training and used only to monitor generalization, it tells us whether the model is overfitting to the training data.

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32)

# 80/20 train/validation split with stratification
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_tensor,
    y_train_tensor,
    test_size=0.2,
    random_state=42,
    stratify=y_train_tensor.numpy()
)

# DataLoaders
BATCH_SIZE = 256

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
train_eval_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")
print(f"Train split size:    {len(X_tr)}")
print(f"Val split size:      {len(X_val)}")
print(f"Test split size:     {len(X_test_tensor)}")

## Step 8: Train the Model

We train MalConv using:
- Loss function: Binary Cross-Entropy (BCELoss) - standard for binary classification with sigmoid output
- Optimizer: Adam with lr=0.001 - adaptive learning rate, generally robust default
- Epochs: 10 - adjust upward if validation loss is still decreasing

Each epoch:
1. Iterates over all training batches, computes loss, backpropagates, updates weights
2. Runs a validation pass (no gradient updates) to track generalization
3. Saves a checkpoint every 5 epochs

In [ ]:
# Initialize model (device defined in Step 2)
model = MalConv().to(device)
import os

print(f"Training on: {device}")

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)

save_dir = SAVE_DIR
os.makedirs(save_dir, exist_ok=True)

NUM_EPOCHS = 5
best_val_loss = float("inf")
best_model_state = None
patience = 2
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    # --- Training phase ---
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    # ---Validation phase-------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs.squeeze(), labels)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- Early stopping check ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        ckpt_path = os.path.join(SAVE_DIR, f'malconv_epoch_{epoch+1}.pt')
        torch.save(model.state_dict(), ckpt_path)
        print(f"  Checkpoint saved to {ckpt_path}")

# Save final model
final_path = os.path.join(SAVE_DIR, 'malconv_final.pt')
torch.save(model.state_dict(), final_path)
print(f"\nFinal model saved to {final_path}")

## Step 9: Evaluate on Test Set

We evaluate the trained model on the held-out test set using four standard classification metrics:

- Accuracy: overall fraction of correct predictions
- Precision: of all files flagged as malware, how many actually are? (minimizes false alarms)
- Recall: of all actual malware files, how many did we catch? (minimizes missed detections)
- F1-Score: harmonic mean of precision and recall - useful when classes are imbalanced
- Confusion Matrix: matrix of TP, FP, TN, FN

Recall is often prioritized here.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels_batch in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)                              # Forward pass
        predicted = (outputs.squeeze() > 0.5).float()       # Threshold at 0.5
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels_batch.numpy())

# Compute metrics
accuracy  = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall    = recall_score(all_labels, all_preds)
f1        = f1_score(all_labels, all_preds)

print("=" * 30)
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print("=" * 30)
print("\nFull Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['Benign', 'Malware']))

# Create confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malware'],
            yticklabels=['Benign', 'Malware'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - MalConv on EMBER 2017 v2')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## Step 10: Save & Load Model - Demonstrated

For Task 2 (SageMaker deployment), we need to be able to save and reload the model weights reliably. Here we demonstrate the full save/load cycle and verify it produces identical outputs.

torch.save(model.state_dict(), path) saves the learned weights as opposed to the entire architecture. The architecture is defined by the MalConv class and must be available at load time.

In [ ]:
# Save final weights
weights_path = os.path.join(SAVE_DIR, 'malconv_final.pt')
torch.save(model.state_dict(), weights_path)
print(f"Model weights saved to: {weights_path}")

# Reload and verify
loaded_model = MalConv().to(device)
loaded_model.load_state_dict(torch.load(weights_path, map_location=device))
loaded_model.eval()
print("Model successfully reloaded from disk.")

# Verify outputs match
sample = X_test_tensor[:4].to(device)
with torch.no_grad():
    orig_out   = model(sample)
    loaded_out = loaded_model(sample)

print(f"\nOriginal outputs: {orig_out.squeeze().cpu().numpy()}")
print(f"Reloaded outputs: {loaded_out.squeeze().cpu().numpy()}")
print(f"Outputs match: {torch.allclose(orig_out, loaded_out)}")

## Step 11: Inference Function

This function accepts a raw PE file (as a file path), extracts its EMBER features, scales them using our fitted scaler, runs inference through the trained model, and returns a human-readable classification result.

In [ ]:
import lief
import ember

def classify_pe_file(file_path: str, model: nn.Module, scaler, device) -> dict:
    """
    Classify a PE file as Malware or Benign using the trained MalConv model.

    Args:
        file_path: Path to the PE (.exe/.dll) file
        model:     Trained MalConv model (in eval mode)
        scaler:    Fitted StandardScaler from training
        device:    torch.device (cuda or cpu)

    Returns:
        dict with 'label' (str), 'probability' (float), and 'confidence' (str)
    """
    # Read raw bytes
    with open(file_path, 'rb') as f:
        bytez = f.read()

    # Extract EMBER features from raw bytes
    extractor = ember.PEFeatureExtractor(feature_version=2)
    features = extractor.feature_vector(bytez)  # Returns numpy array of shape (2381,)

    # Scale using the same scaler fitted during training
    features_scaled = scaler.transform(features.reshape(1, -1)).astype(np.float32)

    # Convert to tensor and run inference
    input_tensor = torch.tensor(features_scaled, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        prob = model(input_tensor).item()

    label = 'Malware' if prob > 0.5 else 'Benign'
    distance = abs(prob - 0.5)
    
    confidence = (
        'High' if distance > 0.4
        else 'Medium' if distance > 0.2
        else 'Low'
    )

    return {
        'label': label,
        'probability': round(prob, 4),
        'confidence': confidence
    }

In [ ]:
import joblib
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler.pkl'))

## Summary

In this notebook we:
1. Vectorized the raw EMBER 2017 v2 JSONL dataset into 2,381-dimensional feature vectors
2. Preprocessed the data using stratified subsampling and StandardScaler normalization
3. Implemented a MalConv-inspired 1D CNN classifier in PyTorch
4. Trained the model for 10 epochs with validation tracking and checkpointing
5. Evaluated on the held-out test set with accuracy, precision, recall, F1, and confusion matrix
6. Demonstrated save/load of model weights
7. Defined an end-to-end classify_pe_file() inference function

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import torch

# Evaluate on train split without shuffling so predictions line up with labels
model.eval()
train_preds = []
train_labels = []

with torch.no_grad():
    for inputs, labels_batch in train_eval_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        predicted = (outputs.squeeze() > 0.5).float()
        train_preds.extend(predicted.cpu().numpy())
        train_labels.extend(labels_batch.numpy())

train_acc = accuracy_score(train_labels, train_preds)

# Re-evaluate test set
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels_batch in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        predicted = (outputs.squeeze() > 0.5).float()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels_batch.numpy())

test_acc = accuracy_score(all_labels, all_preds)

# Negative-control check: shuffle the labels and see if the score collapses
rng = np.random.default_rng(42)
shuffled_labels = rng.permutation(np.array(all_labels))
shuffled_acc = accuracy_score(shuffled_labels, all_preds)

print(f"Train accuracy:    {train_acc:.4f}")
print(f"Test accuracy:     {test_acc:.4f}")
print(f"Train-test gap:    {abs(train_acc - test_acc):.4f}")
print(f"Shuffled baseline:  {shuffled_acc:.4f}")